In [5]:
!python -m spacy download fr_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/fr_core_news_sm-3.8.0/fr_core_news_sm-3.8.0-py3-none-any.whl (16.3 MB)
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


In [11]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 4.1 MB/s eta 0:00:01
   ------------------------------------ --- 1.8/2.0 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.8 MB/s eta 0:00:00


In [12]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker("fr_FR")


def generate_synthetic_survey_data(
    n_rows: int = 200,
    output_file: str = "synthetic_survey.xlsx"
):
    """
    Génère un fichier Excel de données synthétiques
    cohérent avec le pipeline VA-AI.
    """

    genres = ["Homme", "Femme", "Autre"]
    academic_levels = ["L1", "L2", "L3", "M1", "M2"]
    campuses = ["Paris", "Bordeaux", "Lyon", "Toulouse"]

    satisfaction_values = [1, 2, 3, 4, 5]

    # Verbatims types
    positive_comments = [
        "Très bonne école, les cours sont intéressants",
        "J'aime beaucoup le campus et l'ambiance générale",
        "Les professeurs sont pédagogues et disponibles",
        "Organisation claire et efficace"
    ]

    negative_comments = [
        "L'organisation est à revoir",
        "Trop de changements de planning",
        "Le professeur Dupont explique mal",
        "Contactez moi à test@example.com pour plus d'informations",
        "Appelez moi au 06 12 34 56 78"
    ]

    neutral_comments = [
        "RAS",
        "Correct",
        "Moyen",
        "",
        "ok"
    ]

    english_comments = [
        "The courses are good but the workload is heavy",
        "I like the campus but the organization could be better"
    ]

    data = []

    for _ in range(n_rows):
        satisfaction = random.choice(satisfaction_values)

        # Choix du type de commentaire
        comment_pool = random.choices(
            [positive_comments, negative_comments, neutral_comments, english_comments],
            weights=[0.4, 0.3, 0.2, 0.1]
        )[0]

        q58 = random.choice(comment_pool)
        q59 = random.choice(comment_pool)

        # Ajout occasionnel de noms
        if random.random() < 0.3:
            q58 += f" Merci à {fake.first_name()} {fake.last_name()}."
        if random.random() < 0.2:
            q59 += f" Responsable : {fake.first_name()}."

        data.append({
            "Genre": random.choice(genres),
            "Academic level": random.choice(academic_levels),
            "Campus": random.choice(campuses),
            "Date de réponse": fake.date_between(
                start_date="-1y",
                end_date="today"
            ),
            "Q1_satisfaction": satisfaction,
            "Q58_comment": q58,
            "Q59_comment": q59
        })

    df = pd.DataFrame(data)
    df.to_excel(output_file, index=False)

    print(f"✅ Données synthétiques générées : {output_file}")
    print(f"   Nombre de lignes : {n_rows}")

    return df


In [13]:
df_synth = generate_synthetic_survey_data(
    n_rows=500,
    output_file="synthetic_enquete_va_ai.xlsx"
)

df_synth.head()


✅ Données synthétiques générées : synthetic_enquete_va_ai.xlsx
   Nombre de lignes : 500


,Genre,Academic level,Campus,Date de réponse,Q1_satisfaction,Q58_comment,Q59_comment
0,Homme,L1,Toulouse,2025-03-11,1,Les professeurs sont pédagogues et disponibles,Organisation claire et efficace
1,Homme,M1,Bordeaux,2025-02-04,1,Correct,RAS
2,Homme,M2,Paris,2025-06-24,1,Organisation claire et efficace Merci à Matthi...,Organisation claire et efficace
3,Femme,M2,Toulouse,2025-08-30,5,Organisation claire et efficace Merci à Océane...,J'aime beaucoup le campus et l'ambiance générale
4,Homme,M2,Bordeaux,2025-06-10,4,Les professeurs sont pédagogues et disponibles...,Organisation claire et efficace


In [10]:
import pandas as pd
import uuid
import logging
from typing import List, Dict
from datetime import datetime


# =========================
# CONFIGURATION LOGGING
# =========================
def setup_logger(log_file: str = "ingestion.log") -> logging.Logger:
    logger = logging.getLogger("VA_AI_INGESTION")
    logger.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    # Handler fichier
    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    # Handler console
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)

    return logger


logger = setup_logger()


# =========================
# OUTILS
# =========================
def excel_col_letter_to_index(letter: str) -> int:
    letter = letter.upper()
    index = 0
    for char in letter:
        index = index * 26 + (ord(char) - ord('A') + 1)
    return index - 1


def normalize_column_name(col: str) -> str:
    col = col.lower()
    col = col.replace(" ", "_")
    col = col.replace("'", "")
    col = col.replace("é", "e").replace("è", "e").replace("ê", "e")
    col = col.replace("à", "a").replace("ç", "c")
    return col


# =========================
# INGESTION
# =========================
def ingest_excel(
    file_path: str,
    satisfaction_cols_letters: List[str],
    comment_cols_letters: List[str],
    metadata_cols_letters: List[str]
) -> Dict[str, pd.DataFrame]:
    """
    Ingestion d'un fichier Excel avec journalisation complète.
    """

    start_time = datetime.now()
    logger.info("Début ingestion fichier : %s", file_path)

    # === Chargement Excel ===
    try:
        df = pd.read_excel(file_path)
    except Exception as e:
        logger.error("Erreur lecture fichier Excel : %s", e)
        raise

    logger.info("Fichier chargé avec %d lignes et %d colonnes",
                df.shape[0], df.shape[1])

    # === Mapping lettres -> colonnes ===
    try:
        all_letters = satisfaction_cols_letters + comment_cols_letters + metadata_cols_letters
        col_map = {
            letter: df.columns[excel_col_letter_to_index(letter)]
            for letter in all_letters
        }
    except Exception as e:
        logger.error("Erreur mapping colonnes (lettres Excel) : %s", e)
        raise

    logger.info("Colonnes sélectionnées : %s", col_map)

    # === Sélection colonnes ===
    df = df[list(col_map.values())].copy()

    # === Normalisation noms colonnes ===
    df.columns = [normalize_column_name(c) for c in df.columns]

    # === Génération ID réponse ===
    df["id_response"] = [str(uuid.uuid4()) for _ in range(len(df))]

    # === Identification colonnes ===
    satisfaction_cols = [normalize_column_name(col_map[l]) for l in satisfaction_cols_letters]
    comment_cols = [normalize_column_name(col_map[l]) for l in comment_cols_letters]
    metadata_cols = [normalize_column_name(col_map[l]) for l in metadata_cols_letters]

    logger.info("Colonnes satisfaction : %s", satisfaction_cols)
    logger.info("Colonnes commentaires : %s", comment_cols)
    logger.info("Colonnes métadonnées : %s", metadata_cols)

    # === Construction responses_raw ===
    responses_raw = df[
        ["id_response"] + metadata_cols + satisfaction_cols + comment_cols
    ].copy()

    logger.info("Table responses_raw créée (%d lignes)", len(responses_raw))

    # === Construction verbatims ===
    verbatims_rows = []
    skipped_empty = 0

    for _, row in df.iterrows():
        for col in comment_cols:
            text = row[col]
            if pd.notna(text) and str(text).strip() != "":
                verbatims_rows.append({
                    "id_response": row["id_response"],
                    "question_col": col,
                    "text": str(text)
                })
            else:
                skipped_empty += 1

    verbatims = pd.DataFrame(verbatims_rows)

    logger.info("Verbatims extraits : %d", len(verbatims))
    logger.info("Commentaires vides ignorés : %d", skipped_empty)

    duration = (datetime.now() - start_time).total_seconds()
    logger.info("Fin ingestion (durée : %.2f secondes)", duration)

    return {
        "responses_raw": responses_raw,
        "verbatims": verbatims
    }


In [18]:
data = ingest_excel(
    file_path="../res/va_ai_synthetic_bdd.xlsx",
    satisfaction_cols_letters=["F", "G"],
    comment_cols_letters=["H", "I"],
    metadata_cols_letters=["A", "B", "C", "D", "E"]
)

responses_raw = data["responses_raw"]
verbatims = data["verbatims"]


2026-01-13 02:09:12,905 | INFO | Début ingestion fichier : ../res/va_ai_synthetic_bdd.xlsx
2026-01-13 02:09:12,970 | INFO | Fichier chargé avec 250 lignes et 9 colonnes
2026-01-13 02:09:12,971 | INFO | Colonnes sélectionnées : {'F': "Q1_Pour chaque item proposé, merci de cocher les cases correspondant à votre niveau d'accord. (Pas du tout d'accord à tout à fait d'accord)._Dans l'ensemble je suis satisfait de l'Efrei", 'G': "Q1_Pour chaque item proposé, merci de cocher les cases correspondant à votre niveau d'accord. (Pas du tout d'accord à tout à fait d'accord)._Je suis satisfait de ma formation à l'Efrei", 'H': "Q58_Quelles seraient vos suggestions d'amélioration pour l'Efrei", 'I': "Q59_Quels autres messages souhaitez-vous adresser à l'Efrei ?", 'A': 'Genre', 'B': 'Academic level', 'C': 'Campus', 'D': 'A répondu', 'E': 'Date de réponse'}
2026-01-13 02:09:12,974 | INFO | Colonnes satisfaction : ['q1_pour_chaque_item_propose,_merci_de_cocher_les_cases_correspondant_a_votre_niveau_dacco

In [5]:
verbatims

,id_response,question_col,text
0,8b75e03f-60c1-4c46-8c60-b8151bdd81f7,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le..."
1,8b75e03f-60c1-4c46-8c60-b8151bdd81f7,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour les événements et conférences et le...
2,bc7362e1-1e5c-4875-b714-73a508d60fb1,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour la..."
3,bc7362e1-1e5c-4875-b714-73a508d60fb1,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour les associations étudiantes et la d...
4,26c8ee19-bc74-4038-bed7-b233aacccc73,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le..."
...,...,...,...
495,ba9c3abf-6fa0-47b0-9145-bac5243cfcf8,q59_quels_autres_messages_souhaitez-vous_adres...,D/A
496,0cbbd497-94dd-46f5-9a22-a91d84578f8b,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le..."
497,0cbbd497-94dd-46f5-9a22-a91d84578f8b,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour les associations étudiantes et l'am...
498,0c19473b-a26d-4b0f-bdbe-6e8cf7e61ed1,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le..."


In [6]:
responses_raw

,id_response,genre,academic_level,campus,a_repondu,date_de_reponse,"q1_pour_chaque_item_propose,_merci_de_cocher_les_cases_correspondant_a_votre_niveau_daccord._(pas_du_tout_daccord_a_tout_a_fait_daccord)._dans_lensemble_je_suis_satisfait_de_lefrei","q1_pour_chaque_item_propose,_merci_de_cocher_les_cases_correspondant_a_votre_niveau_daccord._(pas_du_tout_daccord_a_tout_a_fait_daccord)._je_suis_satisfait_de_ma_formation_a_lefrei",q58_quelles_seraient_vos_suggestions_damelioration_pour_lefrei,q59_quels_autres_messages_souhaitez-vous_adresser_a_lefrei_?
0,8b75e03f-60c1-4c46-8c60-b8151bdd81f7,H,Bachelor 1,Villejuif,oui,09/12/2025,Plutôt d'accord,Plutôt d'accord,"Globalement je suis satisfait, surtout pour le...",Merci pour les événements et conférences et le...
1,bc7362e1-1e5c-4875-b714-73a508d60fb1,F,Ingénieur 1A,Bordeaux,oui,06/10/2025,Tout à fait d'accord,Tout à fait d'accord,"Globalement je suis satisfait, surtout pour la...",Merci pour les associations étudiantes et la d...
2,26c8ee19-bc74-4038-bed7-b233aacccc73,H,M1,Villejuif,oui,18/11/2025,Plutôt d'accord,Plutôt d'accord,"Globalement je suis satisfait, surtout pour le...",Merci pour les opportunités à l'international ...
3,10384c3b-294c-4ace-8d7b-aefd5ddd75f2,H,Bachelor 3,Bordeaux,oui,08/10/2025,Plutôt d'accord,Plutôt d'accord,"Globalement je suis satisfait, surtout pour l'...",Merci pour la disponibilité de certains enseig...
4,da50546b-30b2-462e-9e55-73566bcd2698,F,Bachelor 2,Bordeaux,oui,14/12/2025,Plutôt pas d'accord,Tout à fait d'accord,Je trouve que les emplois du temps et les déla...,J'aimerais vraiment que l'école progresse sur ...
...,...,...,...,...,...,...,...,...,...,...
245,2699f029-7f1e-452d-bb72-d85571318dbe,F,L3,Bordeaux,oui,11/12/2025,Tout à fait d'accord,Tout à fait d'accord,"Globalement je suis satisfait, surtout pour le...",Merci pour les opportunités à l'international ...
246,1ac05b0d-75c7-49c9-a0c3-bb037ef62664,F,Bachelor 1,Villejuif,oui,21/10/2025,Plutôt d'accord,Plutôt d'accord,"Globalement je suis satisfait, surtout pour le...",Merci pour l'ambiance sur le campus et l'accom...
247,ba9c3abf-6fa0-47b0-9145-bac5243cfcf8,F,Ingénieur 2A,Bordeaux,oui,16/12/2025,Tout à fait d'accord,Tout à fait d'accord,D/A,D/A
248,0cbbd497-94dd-46f5-9a22-a91d84578f8b,H,Bachelor 3,Villejuif,oui,16/11/2025,Tout à fait d'accord,Tout à fait d'accord,"Globalement je suis satisfait, surtout pour le...",Merci pour les associations étudiantes et l'am...


In [3]:
import pandas as pd
import logging
import re
from langdetect import detect, DetectorFactory
from datetime import datetime

DetectorFactory.seed = 0  # reproductibilité


# =========================
# CONFIGURATION LOGGING
# =========================
def setup_logger(log_file: str = "preprocessing.log") -> logging.Logger:
    logger = logging.getLogger("VA_AI_PREPROCESSING")
    logger.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)

    return logger


logger = setup_logger()


# =========================
# REGEX ANONYMISATION
# =========================
EMAIL_REGEX = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
PHONE_REGEX = r"\b(\+?\d{1,3}[\s.-]?)?(\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{3,4}\b"


# =========================
# OUTILS TEXTE
# =========================
def clean_text(text: str) -> str:
    """
    Nettoyage syntaxique du texte (sans perte sémantique).
    """
    text = text.lower()
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\sàâäéèêëîïôöùûüç'-]", "", text)
    return text.strip()


def anonymize_text(text: str) -> str:
    """
    Masque les données identifiantes.
    """
    text = re.sub(EMAIL_REGEX, "[EMAIL]", text)
    text = re.sub(PHONE_REGEX, "[PHONE]", text)
    return text


def detect_language_safe(text: str) -> str:
    """
    Détection robuste de langue.
    """
    try:
        return detect(text)
    except Exception:
        return "unknown"


def is_exploitable(text: str, min_words: int = 3) -> bool:
    """
    Filtrage déterministe des textes non exploitables.
    """
    if not text:
        return False
    tokens = text.split()
    if len(tokens) < min_words:
        return False
    if text in {"ras", "rien", "na", "n/a", "ok", "."}:
        return False
    return True


# =========================
# PRETRAITEMENT
# =========================
def preprocess_verbatims(
    verbatims: pd.DataFrame,
    min_words: int = 3,
    allowed_languages: tuple = ("fr", "en")
) -> pd.DataFrame:
    """
    Prétraitement complet des verbatims.
    """

    start_time = datetime.now()
    logger.info("Début prétraitement")
    logger.info("Verbatims entrants : %d", len(verbatims))

    processed_rows = []

    rejected_short = 0
    rejected_lang = 0
    anonymized_count = 0

    for _, row in verbatims.iterrows():
        raw_text = str(row["text"])

        cleaned = clean_text(raw_text)
        anonymized = anonymize_text(cleaned)

        if anonymized != cleaned:
            anonymized_count += 1

        language = detect_language_safe(anonymized)
        word_count = len(anonymized.split())

        if not is_exploitable(anonymized, min_words=min_words):
            rejected_short += 1
            continue

        if language not in allowed_languages:
            rejected_lang += 1
            continue

        processed_rows.append({
            "id_response": row["id_response"],
            "question_col": row["question_col"],
            "text_raw": raw_text,
            "text_clean": anonymized,
            "language": language,
            "word_count": word_count,
            "is_valid": True
        })

    result = pd.DataFrame(processed_rows)

    logger.info("Verbatims rejetés (vides / trop courts) : %d", rejected_short)
    logger.info("Verbatims rejetés (langue non autorisée) : %d", rejected_lang)
    logger.info("Verbatims anonymisés : %d", anonymized_count)
    logger.info("Verbatims valides pour NLP : %d", len(result))

    duration = (datetime.now() - start_time).total_seconds()
    logger.info("Fin prétraitement (durée : %.2f secondes)", duration)

    return result



In [12]:
detect_language_safe("ascrtgfdvtypbth")

'en'

In [19]:
verbatims_preprocessed = preprocess_verbatims(verbatims)

verbatims_preprocessed


2026-01-13 02:09:21,874 | INFO | Début prétraitement
2026-01-13 02:09:21,875 | INFO | Verbatims entrants : 500
2026-01-13 02:09:22,773 | INFO | Verbatims rejetés (vides / trop courts) : 109
2026-01-13 02:09:22,774 | INFO | Verbatims rejetés (langue non autorisée) : 0
2026-01-13 02:09:22,775 | INFO | Verbatims anonymisés : 0
2026-01-13 02:09:22,776 | INFO | Verbatims valides pour NLP : 391
2026-01-13 02:09:22,777 | INFO | Fin prétraitement (durée : 0.90 secondes)


,id_response,question_col,text_raw,text_clean,language,word_count,is_valid
0,5120d5e8-ea99-4279-88a5-8f20f9f43fa4,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le...",globalement je suis satisfait surtout pour les...,fr,33,True
1,5120d5e8-ea99-4279-88a5-8f20f9f43fa4,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour les événements et conférences et le...,merci pour les événements et conférences et le...,fr,30,True
2,67b12b80-2286-4662-8b53-d7baaf8ea13f,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour la...",globalement je suis satisfait surtout pour la ...,fr,32,True
3,67b12b80-2286-4662-8b53-d7baaf8ea13f,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour les associations étudiantes et la d...,merci pour les associations étudiantes et la d...,fr,27,True
4,41a19f3b-04dd-40ea-8d5a-968326173585,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le...",globalement je suis satisfait surtout pour les...,fr,32,True
...,...,...,...,...,...,...,...
386,c8999bec-909e-472d-92d4-ad615e4703f1,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour l'ambiance sur le campus et l'accom...,merci pour l'ambiance sur le campus et l'accom...,fr,26,True
387,e877f7e2-deac-4195-b51c-8c3d0dd5f8a8,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le...",globalement je suis satisfait surtout pour les...,fr,30,True
388,e877f7e2-deac-4195-b51c-8c3d0dd5f8a8,q59_quels_autres_messages_souhaitez-vous_adres...,Merci pour les associations étudiantes et l'am...,merci pour les associations étudiantes et l'am...,fr,31,True
389,2bd3c414-a4a3-409e-ba9d-c95fbbdbd89a,q58_quelles_seraient_vos_suggestions_dameliora...,"Globalement je suis satisfait, surtout pour le...",globalement je suis satisfait surtout pour les...,fr,31,True


In [ ]:
verbatims_preprocessed

In [13]:
data = ingest_excel(
    file_path="synthetic_enquete_va_ai.xlsx",
    satisfaction_cols_letters=["E"],
    comment_cols_letters=["F", "G"],
    metadata_cols_letters=["A", "B", "C", "D"]
)

responses_raw = data["responses_raw"]
verbatims = data["verbatims"]
verbatims

2026-01-12 22:10:50,291 | INFO | Début ingestion fichier : synthetic_enquete_va_ai.xlsx
2026-01-12 22:10:50,344 | INFO | Fichier chargé avec 500 lignes et 7 colonnes
2026-01-12 22:10:50,344 | INFO | Colonnes sélectionnées : {'E': 'Q1_satisfaction', 'F': 'Q58_comment', 'G': 'Q59_comment', 'A': 'Genre', 'B': 'Academic level', 'C': 'Campus', 'D': 'Date de réponse'}
2026-01-12 22:10:50,346 | INFO | Colonnes satisfaction : ['q1_satisfaction']
2026-01-12 22:10:50,347 | INFO | Colonnes commentaires : ['q58_comment', 'q59_comment']
2026-01-12 22:10:50,347 | INFO | Colonnes métadonnées : ['genre', 'academic_level', 'campus', 'date_de_reponse']
2026-01-12 22:10:50,348 | INFO | Table responses_raw créée (500 lignes)
2026-01-12 22:10:50,358 | INFO | Verbatims extraits : 958
2026-01-12 22:10:50,358 | INFO | Commentaires vides ignorés : 42
2026-01-12 22:10:50,359 | INFO | Fin ingestion (durée : 0.07 secondes)


,id_response,question_col,text
0,89e8b28e-e0df-4eb4-9416-1c8c7b121682,q58_comment,Les professeurs sont pédagogues et disponibles
1,89e8b28e-e0df-4eb4-9416-1c8c7b121682,q59_comment,Organisation claire et efficace
2,1df169cd-8cfd-4401-971f-95a86c03ae5f,q58_comment,Correct
3,1df169cd-8cfd-4401-971f-95a86c03ae5f,q59_comment,RAS
4,fdb3feed-ca6f-4593-af54-c6987ab0bc50,q58_comment,Organisation claire et efficace Merci à Matthi...
...,...,...,...
953,6daa2e15-50ac-4af1-a8f2-7cf10a9ebbd3,q59_comment,Moyen
954,2619a40f-b928-4de8-a784-42609a11e776,q58_comment,Moyen Merci à Alain Tanguy.
955,2619a40f-b928-4de8-a784-42609a11e776,q59_comment,RAS
956,d23f1c6f-36c6-41e5-8e5b-e26f504d7f4b,q58_comment,Correct


In [14]:
verbatims_preprocessed = preprocess_verbatims(verbatims)

verbatims_preprocessed

2026-01-12 22:10:59,831 | INFO | Début prétraitement
2026-01-12 22:10:59,831 | INFO | Verbatims entrants : 958
2026-01-12 22:11:07,589 | INFO | Rejetés (vides / courts) : 113
2026-01-12 22:11:07,590 | INFO | Rejetés (langue) : 92
2026-01-12 22:11:07,590 | INFO | Emails anonymisés : 62 (verbatims: 62)
2026-01-12 22:11:07,590 | INFO | Téléphones anonymisés : 72 (verbatims: 72)
2026-01-12 22:11:07,591 | INFO | Noms anonymisés (NER BERT) : 208 (verbatims: 208)
2026-01-12 22:11:07,591 | INFO | Verbatims valides NLP : 753
2026-01-12 22:11:07,592 | INFO | Fin prétraitement – durée 7.76s


,id_response,question_col,text_raw,text_clean,language,word_count,is_valid
0,89e8b28e-e0df-4eb4-9416-1c8c7b121682,q58_comment,Les professeurs sont pédagogues et disponibles,les professeurs sont pédagogues et disponibles,fr,6,True
1,89e8b28e-e0df-4eb4-9416-1c8c7b121682,q59_comment,Organisation claire et efficace,organisation claire et efficace,fr,4,True
2,fdb3feed-ca6f-4593-af54-c6987ab0bc50,q58_comment,Organisation claire et efficace Merci à Matthi...,organisation claire et efficace merci à name,fr,7,True
3,fdb3feed-ca6f-4593-af54-c6987ab0bc50,q59_comment,Organisation claire et efficace,organisation claire et efficace,fr,4,True
4,e13a45b3-62d3-4114-9f24-97ba1aa6b332,q58_comment,Organisation claire et efficace Merci à Océane...,organisation claire et efficace merci à océane...,fr,8,True
...,...,...,...,...,...,...,...
748,eee94558-15c8-4b74-b35f-8b684589c89e,q59_comment,Organisation claire et efficace,organisation claire et efficace,fr,4,True
749,1ba3dd3b-724a-48b5-b8bf-caa5fc0d370e,q58_comment,J'aime beaucoup le campus et l'ambiance générale,j'aime beaucoup le campus et l'ambiance générale,fr,7,True
750,1ba3dd3b-724a-48b5-b8bf-caa5fc0d370e,q59_comment,"Très bonne école, les cours sont intéressants",très bonne école les cours sont intéressants,fr,7,True
751,f73d1f4f-e4f1-4c87-9d92-5ad412424b01,q58_comment,"Très bonne école, les cours sont intéressants",très bonne école les cours sont intéressants,fr,7,True


In [30]:
import pandas as pd
import logging
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.nn.functional import softmax
from datetime import datetime


# =========================
# LOGGING
# =========================
def setup_logger(log_file: str = "sentiment_analysis.log") -> logging.Logger:
    logger = logging.getLogger("VA_AI_SENTIMENT")
    logger.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)

    return logger


logger = setup_logger()


# =========================
# CHARGEMENT MODELE
# =========================
MODEL_NAME = "ac0hik/Sentiment_Analysis_French"

logger.info("Chargement du modèle de sentiment : %s", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()


# =========================
# MAPPING SENTIMENT
# =========================
def map_star_rating_to_sentiment(stars: int) -> str:
    """
    Mapping 1–5 étoiles vers sentiment lisible.
    """
    if stars <= 2:
        return "negative"
    elif stars == 3:
        return "neutral"
    else:
        return "positive"


# =========================
# SENTIMENT ANALYSIS
# =========================
def sentiment_analysis(
    verbatims_preprocessed: pd.DataFrame,
    batch_size: int = 16,
    max_length: int = 256
) -> pd.DataFrame:
    """
    Analyse de sentiment batch avec BERT nlptown.
    """

    start_time = datetime.now()
    logger.info("Début sentiment analysis")
    logger.info("Verbatims à analyser : %d", len(verbatims_preprocessed))

    texts = verbatims_preprocessed["text_clean"].tolist()
    sentiment_labels = []
    sentiment_scores = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            outputs = model(**encoded)
            probs = softmax(outputs.logits, dim=1)

            for prob in probs:
                score, star_class = torch.max(prob, dim=0)
                stars = star_class.item() + 1  # classes 0–4 → 1–5

                sentiment_labels.append(
                    map_star_rating_to_sentiment(stars)
                )
                sentiment_scores.append(score.item())

    result = verbatims_preprocessed.copy()
    result["sentiment_label"] = sentiment_labels
    result["sentiment_score"] = sentiment_scores

    duration = (datetime.now() - start_time).total_seconds()
    logger.info("Fin sentiment analysis (%.2f secondes)", duration)

    return result


2026-01-13 17:18:32,444 | INFO | Chargement du modèle de sentiment : ac0hik/Sentiment_Analysis_French


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/922 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

In [31]:
verbatims_with_sentiment = sentiment_analysis(verbatims_preprocessed)

verbatims_with_sentiment[
    ["text_clean", "sentiment_label", "sentiment_score"]
]


2026-01-13 17:19:27,108 | INFO | Début sentiment analysis
2026-01-13 17:19:27,111 | INFO | Verbatims à analyser : 391
2026-01-13 17:19:31,677 | INFO | Fin sentiment analysis (4.57 secondes)


,text_clean,sentiment_label,sentiment_score
0,globalement je suis satisfait surtout pour les...,neutral,0.856607
1,merci pour les événements et conférences et le...,neutral,0.856810
2,globalement je suis satisfait surtout pour la ...,neutral,0.854885
3,merci pour les associations étudiantes et la d...,neutral,0.854508
4,globalement je suis satisfait surtout pour les...,neutral,0.856827
...,...,...,...
386,merci pour l'ambiance sur le campus et l'accom...,neutral,0.857641
387,globalement je suis satisfait surtout pour les...,neutral,0.857387
388,merci pour les associations étudiantes et l'am...,neutral,0.857172
389,globalement je suis satisfait surtout pour les...,neutral,0.857299


In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan scikit-learn

   ---------------------------------------- 0.0/671.7 kB ? eta -:--:--
   --------------- ------------------------ 262.1/671.7 kB ? eta -:--:--
   ---------------------------------------- 671.7/671.7 kB 2.3 MB/s eta 0:00:00

   ---------------- ----------------------- 2/5 [umap-learn]
   ------------------------ --------------- 3/5 [sentence-transformers]
   ------------------------ --------------- 3/5 [sentence-transformers]
   -------------------------------- ------- 4/5 [bertopic]
   ---------------------------------------- 5/5 [bertopic]



In [28]:
import pandas as pd
import logging
from datetime import datetime
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer


# =========================
# LOGGING
# =========================
def setup_logger(log_file: str = "topic_modeling.log") -> logging.Logger:
    logger = logging.getLogger("VA_AI_TOPIC")
    logger.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)

    return logger


logger = setup_logger()


# =========================
# TOPIC MODELING
# =========================
def topic_modeling_bertopic(
    verbatims_with_sentiment: pd.DataFrame,
    embedding_model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
    min_topic_size: int = 15,
    n_gram_range: tuple = (1, 2)
) -> pd.DataFrame:
    """
    Topic modeling avec BERTopic.
    """

    start_time = datetime.now()
    logger.info("Début topic modeling (BERTopic)")
    logger.info("Verbatims à traiter : %d", len(verbatims_with_sentiment))

    texts = verbatims_with_sentiment["text_clean"].tolist()

    # === Embeddings ===
    logger.info("Chargement du modèle d'embeddings : %s", embedding_model_name)
    embedding_model = SentenceTransformer(embedding_model_name)

    # === BERTopic ===
    topic_model = BERTopic(
        embedding_model=embedding_model,
        min_topic_size=min_topic_size,
        n_gram_range=n_gram_range,
        language="multilingual",
        calculate_probabilities=True,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(texts)

    logger.info("Nombre de topics détectés (hors outliers) : %d",
                len(set(t for t in topics if t != -1)))

    # === Récupération labels ===
    topic_labels = {
        topic_id: topic_model.get_topic_info().loc[
            topic_model.get_topic_info()["Topic"] == topic_id, "Name"
        ].values[0]
        for topic_id in set(topics)
    }

    # === Construction sortie ===
    result = verbatims_with_sentiment.copy()
    result["topic_id"] = topics
    result["topic_label"] = result["topic_id"].map(topic_labels)
    result["topic_score"] = [
        probs[i][topic] if topic != -1 else 0.0
        for i, topic in enumerate(topics)
    ]

    duration = (datetime.now() - start_time).total_seconds()
    logger.info("Fin topic modeling (%.2f secondes)", duration)

    return result, topic_model


In [29]:
verbatims_with_topics, topic_model = topic_modeling_bertopic(
    verbatims_with_sentiment
)

verbatims_with_topics[
    ["text_clean", "topic_id", "topic_label", "topic_score"]
]


2026-01-13 09:37:57,816 | INFO | Début topic modeling (BERTopic)
2026-01-13 09:37:57,816 | INFO | Verbatims à traiter : 391
2026-01-13 09:37:57,817 | INFO | Chargement du modèle d'embeddings : paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-01-13 09:38:44,487 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

2026-01-13 09:38:44,881 - BERTopic - Embedding - Completed ✓
2026-01-13 09:38:44,882 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-13 09:38:50,183 - BERTopic - Dimensionality - Completed ✓
2026-01-13 09:38:50,185 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-13 09:38:50,200 - BERTopic - Cluster - Completed ✓
2026-01-13 09:38:50,202 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-13 09:38:50,218 - BERTopic - Representation - Completed ✓
2026-01-13 09:38:50,237 | INFO | Nombre de topics détectés (hors outliers) : 8
2026-01-13 09:38:50,255 | INFO | Fin topic modeling (52.44 secondes)


,text_clean,topic_id,topic_label,topic_score
0,globalement je suis satisfait surtout pour les...,0,0_de_et_les_pour,0.839212
1,merci pour les événements et conférences et le...,0,0_de_et_les_pour,0.820937
2,globalement je suis satisfait surtout pour la ...,0,0_de_et_les_pour,0.851753
3,merci pour les associations étudiantes et la d...,3,3_de moderniser_certaines_ventilation_écrans,1.000000
4,globalement je suis satisfait surtout pour les...,0,0_de_et_les_pour,0.823707
...,...,...,...,...
386,merci pour l'ambiance sur le campus et l'accom...,0,0_de_et_les_pour,0.869502
387,globalement je suis satisfait surtout pour les...,0,0_de_et_les_pour,0.549618
388,merci pour les associations étudiantes et l'am...,0,0_de_et_les_pour,0.809923
389,globalement je suis satisfait surtout pour les...,0,0_de_et_les_pour,1.000000


In [13]:
pip install detoxify

Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip install -U transformers

In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import pandas as pd
import os

# Load model and tokenizer once
tokenizer = AutoTokenizer.from_pretrained("Jean-Baptiste/camembert-ner-with-dates")
model = AutoModelForTokenClassification.from_pretrained("Jean-Baptiste/camembert-ner-with-dates")

# Create NER pipeline
nlp = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy="simple")

def extract_named_entities(text):
    """
    Extract named entities from French text using CamemBERT NER model.
    
    Args:
        text (str): Input French text
        
    Returns:
        pd.DataFrame: DataFrame with entity information (entity, label, score)
        list: Raw NER results
    """
    results = nlp(text)
    
    # Convert to DataFrame for better visualization
    if results:
        df_results = pd.DataFrame(results)
        return df_results, results
    else:
        return pd.DataFrame(), results

# Test examples
test_texts = [
    "Apple est créée le 1er avril 1976 dans le garage de la maison d'enfance de Dupont francois à Los Altos en Californie par Steve Jobs, Steve Wozniak et Ronald Wayne14, puis constituée sous forme de société le 3 janvier 1977 à l'origine sous le nom d'Apple Computer, mais pour ses 30 ans et pour refléter la diversification de ses produits, le mot « computer » est retiré le 9 janvier 2015.",
    "Jean Dupont travaille chez Microsoft à Paris depuis le 15 mars 2020.",
    "La réunion aura lieu le 25 décembre à Lyon.",
    "voici mon email: idriss.dag@gmail.com et mon numéro de téléphone: +33 6 12 34 56 78.",
    "Le président Emmanuel Macron a rencontré Angela Merkel à Berlin le 10 juillet 2021.",
    "La Tour Eiffel est située à Paris et a été construite en 1889 par Gustave Eiffel.",
    "Contactez-moi à l'adresse idriss.dag@gmail.com ou par téléphone au +33 6 12 34 56 78.",
    "Le 14 juillet, la France célèbre sa fête nationale avec des feux d'artifice à travers le pays.",
    "Marie Curie est née à Varsovie en Pologne en 1867 et a remporté deux prix Nobel. L'url de son site est http://www.mariecurie.org.",
    "Le 11 septembre 2001, les États-Unis ont été frappés par des attaques terroristes coordonnées."
]

for text in test_texts:
    print(f"\n📝 Texte: {text[:80]}...")
    df, raw = extract_named_entities(text)
    print(df)

Device set to use cuda:0



📝 Texte: Apple est créée le 1er avril 1976 dans le garage de la maison d'enfance de Dupon...
   entity_group     score                       word  start  end
0           ORG  0.978720                      Apple      0    5
1          DATE  0.981889  le 1er avril 1976 dans le     15   41
2           PER  0.892213            Dupont francois     74   90
3           LOC  0.994926                  Los Altos     92  102
4           LOC  0.995211                 Californie    105  116
5           PER  0.996297                 Steve Jobs    120  131
6           PER  0.996287              Steve Wozniak    132  146
7           PER  0.996054               Ronald Wayne    149  162
8          DATE  0.994111        le 3 janvier 1977 à    203  223
9           ORG  0.974233           d'Apple Computer    245  262
10         DATE  0.992573                  30 ans et    277  287
11         DATE  0.993486         le 9 janvier 2015.    368  387

📝 Texte: Jean Dupont travaille chez Microsoft à Paris depuis